In [3]:
"""
Data Analysis and Visualisation
Data cleaning script for Assignment_2_Data.xlsx

Two integrity problems were identified during data understanding:
  1. Duplicate store records (the company warned records may be duplicated)
  2. Inconsistent naming in the 'Top selling product' field
This script fixes both, adds analysis fields, and writes a cleaning log.

Authur: Abdulrahman Balogun

"""
import pandas as pd

# ---------------------------------------------------------------------------
# Load raw data
# ---------------------------------------------------------------------------
df = pd.read_excel("Assignment_2_Data.xlsx")
raw_rows = len(df)
print(f"Raw rows: {raw_rows}  |  Unique Store #: {df['Store #'].nunique()}")

# ---------------------------------------------------------------------------
# Remove exact duplicate rows
# Match on the WHOLE row (all 25 columns), not just Store #, so only genuine
# copies are removed. keep='first' retains one copy of each store.
# ---------------------------------------------------------------------------
df = df.drop_duplicates(keep="first").reset_index(drop=True)
removed = raw_rows - len(df)
assert df["Store #"].is_unique, "Store # should be unique after de-duplication"
print(f"Removed {removed} duplicate rows  ->  {len(df)} unique stores")

# ---------------------------------------------------------------------------
# Standardise 'Top selling product'
# Order matters: strip whitespace first, THEN map variants, otherwise labels
# with a trailing space (e.g. 'Home electronics ') won't match.
# ---------------------------------------------------------------------------
df["Top selling product"] = df["Top selling product"].str.strip()
df["Top selling product"] = df["Top selling product"].replace({
    "tablet": "Tablets",   # lower-case variant
    "TV'S":   "TVs",        # inconsistent casing/punctuation
})
print(f"Product categories after cleaning: {sorted(df['Top selling product'].unique())}")

# ---------------------------------------------------------------------------
# Add derived fields to support the analysis
# ---------------------------------------------------------------------------
df["Profit margin %"]       = (df["Total profit (£)"] / df["Total sales (£)"] * 100).round(1)
df["Sales per employee (£)"] = (df["Total sales (£)"] / df["Number of employees"]).round(0)
df["Q1 to Q4 growth %"]      = ((df["Store Sales (Q4) £"] - df["Store Sales (Q1) £"])
                                / df["Store Sales (Q1) £"] * 100).round(1)

# ---------------------------------------------------------------------------
# Save cleaned data + a cleaning log on a second sheet
# ---------------------------------------------------------------------------
log = pd.DataFrame({
    "Issue found":  ["Exact duplicate rows (Norwich x6; stores 300-307 x2)",
                     "'tablet' vs 'Tablets'",
                     "\"TV'S\" casing",
                     "Trailing spaces in product labels"],
    "Action taken": [f"Removed {removed} exact duplicates ({raw_rows} -> {len(df)})",
                     "Mapped 'tablet' -> 'Tablets'",
                     "Standardised \"TV'S\" -> 'TVs'",
                     "Stripped whitespace and merged into correct category"],
})

with pd.ExcelWriter("Store_Data_CLEANED.xlsx", engine="openpyxl") as w:
    df.to_excel(w, sheet_name="Cleaned Data", index=False)
    log.to_excel(w, sheet_name="Cleaning Log", index=False)

print("Saved Store_Data_CLEANED.xlsx")

Raw rows: 693  |  Unique Store #: 680
Removed 13 duplicate rows  ->  680 unique stores
Product categories after cleaning: ['Audio equipment', 'Cameras', 'Home electronics', 'Mobile phones', 'TVs', 'Tablets']
Saved Store_Data_CLEANED.xlsx


In [7]:
# Case study - data cleaning, analysis and regression

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

navy = "#1a3a5c"
orange = "#e8743b"
grey = "#8a93a0"
red = "#c0392b"

plt.rcParams["figure.dpi"] = 150
os.makedirs("figs", exist_ok=True)


# ---------- step 1: load and check the data ----------

df = pd.read_excel("Assignment_2_Data.xlsx")

print("rows and columns:", df.shape)
print("duplicate rows:", df.duplicated().sum())
print("missing values:", df.isnull().sum().sum())

# see which stores got entered more than once
dupes = df[df.duplicated(subset=["Store #", "City"], keep=False)]
print(dupes[["Store #", "City"]].value_counts())

# product names are a mess - tablet vs Tablets, TV'S, extra spaces
print(df["Top selling product"].unique())

# one column header has a space at the end
print([c for c in df.columns if c != c.strip()])

# check the totals actually add up before trusting anything
qsum = df[["Store Sales (Q1) £", "Store Sales (Q2) £",
           "Store Sales (Q3) £", "Store Sales (Q4) £"]].sum(axis=1)
print("quarters match total sales:", (qsum - df["Total sales (£)"]).abs().max() == 0)

check = df["Total sales (£)"] - df["Total yearly overhead"] - df["Total profit (£)"]
print("profit = sales - overhead:", check.abs().max() == 0)

# how much the duplicates inflate sales by
extra = df["Total sales (£)"].sum() - df.drop_duplicates()["Total sales (£)"].sum()
print("sales inflated by duplicates: £", round(extra))


# ---------- step 2: cleaning ----------

# tidy up headers and text values
df.columns = [c.strip() for c in df.columns]
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip()

# drop the 13 duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("removed", before - len(df), "duplicates, now", len(df), "stores")

# fix the product names
df["Top selling product"] = df["Top selling product"].replace({
    "tablet": "Tablets",
    "TV'S": "TVs",
})
print(sorted(df["Top selling product"].unique()))

# extra columns I need for the analysis
df["Profit margin %"] = (df["Total profit (£)"] / df["Total sales (£)"] * 100).round(1)
df["Q1_to_Q4_growth_%"] = ((df["Store Sales (Q4) £"] - df["Store Sales (Q1) £"])
                           / df["Store Sales (Q1) £"] * 100).round(1)
df["Sales per employee"] = (df["Total sales (£)"] / df["Number of employees"]).round(0)
df["Late shipment rate %"] = (df["Total late shipments"]
                              / df["Total online purchases fulfilled"] * 100).round(2)

# tableau needs the quarters stacked in one column for the trend chart
long = df.melt(id_vars=["Store #", "City"],
               value_vars=["Store Sales (Q1) £", "Store Sales (Q2) £",
                           "Store Sales (Q3) £", "Store Sales (Q4) £"],
               var_name="Quarter", value_name="Quarterly Sales")
long["Quarter"] = long["Quarter"].str.extract(r"(Q\d)")

# save the cleaned file for tableau
with pd.ExcelWriter("Cleaned_Store_Data.xlsx") as xl:
    df.to_excel(xl, sheet_name="Cleaned Store Data", index=False)
    long.to_excel(xl, sheet_name="Quarterly Sales Long", index=False)
print("saved Cleaned_Store_Data.xlsx")


# ---------- step 3: answering the questions ----------

# best and worst stores
print("\ntop 5 sales")
print(df.nlargest(5, "Total sales (£)")[["City", "Total sales (£)", "Total profit (£)"]])
print("\nbottom 5 sales")
print(df.nsmallest(5, "Total sales (£)")[["City", "Total sales (£)", "Total profit (£)"]])

loss = df[df["Total profit (£)"] < 0].sort_values("Total profit (£)")
print("\nstores losing money:", len(loss))
print(loss[["City", "Total sales (£)", "Total profit (£)"]])

# figure 1 - top and bottom 10
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
t10 = df.nlargest(10, "Total sales (£)").sort_values("Total sales (£)")
b10 = df.nsmallest(10, "Total sales (£)").sort_values("Total sales (£)", ascending=False)
ax[0].barh(t10["City"], t10["Total sales (£)"] / 1e6, color=navy)
ax[0].set_title("Top 10 stores by total sales")
ax[0].set_xlabel("Total sales (£m)")
ax[1].barh(b10["City"], b10["Total sales (£)"] / 1e6, color=grey)
ax[1].set_title("Bottom 10 stores by total sales")
ax[1].set_xlabel("Total sales (£m)")
plt.tight_layout()
plt.savefig("figs/fig1_top_bottom_sales.png")
plt.close()

# figure 2 - the loss making stores
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(loss["City"], loss["Total profit (£)"] / 1e6, color=red)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Loss-making stores (6 of 680)")
ax.set_xlabel("Total profit (£m)")
plt.tight_layout()
plt.savefig("figs/fig2_losses.png")
plt.close()

# do more staff = more sales? london left out because its an outlier
d2 = df[df["City"] != "London"]
r1 = d2["Number of employees"].corr(d2["Total sales (£)"])
r2 = d2["Number of employees"].corr(d2["Total profit (£)"])
print("\nemployees vs sales r =", round(r1, 2))
print("employees vs profit r =", round(r2, 2), "- negative, more staff less profit")

# figure 3
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(d2["Number of employees"], d2["Total sales (£)"] / 1e6, s=12, alpha=0.45, color=navy)
ax[0].set_xlabel("Number of employees")
ax[0].set_ylabel("Total sales (£m)")
ax[0].set_title("Employees vs sales (r = %.2f)" % r1)
ax[1].scatter(d2["Number of employees"], d2["Total profit (£)"] / 1e6, s=12, alpha=0.45, color=orange)
ax[1].set_xlabel("Number of employees")
ax[1].set_ylabel("Total profit (£m)")
ax[1].set_title("Employees vs profit (r = %.2f)" % r2)
plt.tight_layout()
plt.savefig("figs/fig3_employees.png")
plt.close()

# best selling products
prod = df.groupby("Top selling product").agg(
    stores=("Store #", "count"),
    units=("Quantity sold", "sum"),
    sales=("Total sales (£)", "sum")).sort_values("units", ascending=False)
print("\n", prod)

# figure 4
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ps = prod.sort_values("units")
ax[0].barh(ps.index, ps["units"] / 1000, color=navy)
ax[0].set_title("Total units sold by top-selling category")
ax[0].set_xlabel("Units (thousands)")
ax[1].barh(ps.index, ps["stores"], color=orange)
ax[1].set_title("Stores where category is the best seller")
ax[1].set_xlabel("Store count")
plt.tight_layout()
plt.savefig("figs/fig4_products.png")
plt.close()

# does store size matter for profit
r3 = d2["Store Size (m²)"].corr(d2["Total profit (£)"])
r4 = d2["City population"].corr(d2["Total sales (£)"])
print("\nsize vs profit r =", round(r3, 2), "- weak")
print("population vs sales r =", round(r4, 2), "- location matters more than size")

# figure 5
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(d2["Store Size (m²)"], d2["Total profit (£)"] / 1e6, s=12, alpha=0.45, color=navy)
z = np.polyfit(d2["Store Size (m²)"], d2["Total profit (£)"] / 1e6, 1)
xs = np.linspace(d2["Store Size (m²)"].min(), d2["Store Size (m²)"].max(), 50)
ax.plot(xs, np.polyval(z, xs), color=orange, lw=2)
ax.set_xlabel("Store size (m²)")
ax.set_ylabel("Total profit (£m)")
ax.set_title("Store size vs profit (r = %.2f)" % r3)
plt.tight_layout()
plt.savefig("figs/fig5_size.png")
plt.close()

# quarterly picture - big spike in Q3 then drops back in Q4
qt = df[["Store Sales (Q1) £", "Store Sales (Q2) £",
         "Store Sales (Q3) £", "Store Sales (Q4) £"]].sum() / 1e9
print("\nquarterly sales (bn):")
print(qt.round(2))

print("\nbiggest Q1 to Q4 growth:")
print(df.nlargest(5, "Q1_to_Q4_growth_%")[["City", "Store Sales (Q1) £",
                                           "Store Sales (Q4) £", "Q1_to_Q4_growth_%"]])

# figure 6 - company total vs london on a second axis
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(["Q1", "Q2", "Q3", "Q4"], qt.values, marker="o", color=navy, lw=2.5, label="All stores (£bn)")
lon = df[df["City"] == "London"][["Store Sales (Q1) £", "Store Sales (Q2) £",
                                  "Store Sales (Q3) £", "Store Sales (Q4) £"]].iloc[0] / 1e6
ax2 = ax.twinx()
ax2.plot(["Q1", "Q2", "Q3", "Q4"], lon.values, marker="s", color=orange, lw=2, ls="--",
         label="London (right axis, £m)")
ax.set_ylabel("Company sales (£bn)")
ax2.set_ylabel("London sales (£m)", color=orange)
ax.set_title("Quarterly sales - company Q3 peak vs London growth")
for i, v in enumerate(qt.values):
    ax.annotate("£%.2fbn" % v, (i, v), textcoords="offset points",
                xytext=(0, -15 if i == 2 else 8), ha="center", fontsize=8.5)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("figs/fig6_quarters.png")
plt.close()

# which city sold the most of each product
champs = df.sort_values("Quantity sold", ascending=False).groupby(
    "Top selling product").first()[["City", "Quantity sold"]]
print("\ntop city for each product:")
print(champs)

# numbers for the dashboard kpi cards
margin = df["Total profit (£)"].sum() / df["Total sales (£)"].sum() * 100
print("\nstores:", len(df))
print("chain sales: £%.2fbn" % (df["Total sales (£)"].sum() / 1e9))
print("chain profit: £%.2fbn" % (df["Total profit (£)"].sum() / 1e9))
print("chain margin: %.1f%%" % margin)
print("loss making:", (df["Total profit (£)"] < 0).sum())


# ---------- step 4: regression on the marketing data ----------

m = pd.read_excel("2 year record.xlsx")
m = m[m["Month"].str.upper() != "TOTAL"].reset_index(drop=True)  # drop the total row
print("\nmonths of data:", len(m))
print("budget range: £%d to £%d" % (m["Marketing budget"].min(), m["Marketing budget"].max()))

X = m[["Marketing budget"]].values
y = m["Store sales"].values

r = np.corrcoef(m["Marketing budget"], y)[0, 1]
print("correlation r =", round(r, 3))

model = LinearRegression().fit(X, y)
pred = model.predict(X)
print("sales = %.2f x budget + (%.2f)" % (model.coef_[0], model.intercept_))
print("r squared =", round(r2_score(y, pred), 4))
print("mae = £%d" % mean_absolute_error(y, pred))

# the 4 budgets the company asked about
budgets = np.array([[20000], [25000], [45000], [50000]])
for b, p in zip(budgets.ravel(), model.predict(budgets)):
    print("£%d budget -> £%d sales" % (b, p))
# note - all 4 are above the max budget in the data (£15,698) so these are
# extrapolated. real returns probably flatten out at higher spend

# figure 8 - the regression line with the predictions marked
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X, y / 1e6, color=navy, s=45, zorder=3, label="Monthly data (24 months)")
xs = np.linspace(0, 52000, 100).reshape(-1, 1)
ax.plot(xs, model.predict(xs) / 1e6, color=orange, lw=2,
        label="Fit: y = %.0fx %+.0f (R² = %.3f)" % (model.coef_[0], model.intercept_, r2_score(y, pred)))
ax.scatter(budgets, model.predict(budgets) / 1e6, color=red, marker="D", s=70,
           zorder=4, label="Predictions")
for b, p in zip(budgets.ravel(), model.predict(budgets)):
    ax.annotate("£%dk -> £%.1fm" % (b / 1000, p / 1e6), (b, p / 1e6),
                textcoords="offset points", xytext=(-10, 10), fontsize=8.5, ha="right")
ax.set_xlabel("Monthly marketing budget (£)")
ax.set_ylabel("Monthly store sales (£m)")
ax.set_title("Marketing budget vs sales - London store")
ax.legend(fontsize=8.5, loc="upper left")
plt.tight_layout()
plt.savefig("figs/fig8_regression.png")
plt.close()

# figure 9 - residuals to check the linear fit is ok
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(pred / 1e6, (y - pred) / 1e6, color=navy, s=40)
ax.axhline(0, color=orange, lw=1.5)
ax.set_xlabel("Fitted sales (£m)")
ax.set_ylabel("Residual (£m)")
ax.set_title("Residual plot")
plt.tight_layout()
plt.savefig("figs/fig9_residuals.png")
plt.close()

print("\ndone - figures saved in figs folder")


rows and columns: (693, 25)
duplicate rows: 13
missing values: 0
Store #  City       
18       Norwich        6
300      Royton         2
301      Clevedon       2
302      Dronfield      2
303      Ossett         2
304      Woodlesford    2
305      Wallington     2
306      Norton         2
307      Leek           2
Name: count, dtype: int64
['Mobile phones' 'Tablets' "TV'S" 'Home electronics ' 'Cameras'
 'Audio equipment ' 'tablet']
['Opening Hours ', 'Total online purchases fulfilled ']
quarters match total sales: True
profit = sales - overhead: True
sales inflated by duplicates: £ 150466639
removed 13 duplicates, now 680 stores
['Audio equipment', 'Cameras', 'Home electronics', 'Mobile phones', 'TVs', 'Tablets']
saved Cleaned_Store_Data.xlsx

top 5 sales
             City  Total sales (£)  Total profit (£)
0          London         38826894          32824986
583       Kirkham         17778873          14879097
530         Nairn         17672265          15251505
401  West Wickham 